# UCSY Myanmar ASR with Whisper

This notebook trains from **`NLP Project1/audio_files/` only**. It filters the existing `asr_work/*_whisper.csv` manifests to `source == "ucsy"`; it never opens audio in `MyanmarASRScriptsAndData/`. The notebook uses human-speech candidates, with synthetic `gemini/` clips held for a separate experiment. Read [README.md](README.md) before running it.

On Windows, follow the PowerShell setup in [README.md](README.md) before opening the notebook. Run cells top to bottom. Keep `QUICK_RUN = True` for a short pipeline check. Then set it to `False`, restart the kernel, and run the full experiment. The test set is evaluated only in full mode after training.


## 1. Install the notebook dependencies

First install PyTorch in this Jupyter kernel using the [official selector](https://pytorch.org/get-started/locally/) for your CPU/GPU. Restart the kernel after installing PyTorch. FFmpeg must also be on `PATH`. This cell installs the remaining packages; the notebook targets Transformers 4.57.1. Restart the kernel after this cell if Jupyter asks.


In [2]:
%pip install -q "transformers==4.57.1" "accelerate>=1.0" "soundfile>=0.12" "jiwer>=3" "tqdm>=4" "numpy>=1.24" "ffmpeg-python>=0.2" "huggingface_hub[hf_xet]"


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Set paths and check hardware

The notebook works when Jupyter starts in either the `audio-data/` root or this `ucsy_asr/` folder. `QUICK_RUN` selects 256 train and 64 validation clips for a 20-step check. Full mode uses all eligible UCSY clips.


In [ ]:
import csv
import json
import random
import shutil
import subprocess
import unicodedata
from pathlib import Path

import jiwer
import numpy as np
import soundfile as sf
import torch
import transformers
from IPython.display import Audio, display
from torch.utils.data import Dataset
from tqdm.auto import tqdm
from transformers import (Seq2SeqTrainer, Seq2SeqTrainingArguments,
                          WhisperForConditionalGeneration, WhisperProcessor)

cwd = Path.cwd().resolve()
ROOT = next(
    (p for p in (cwd, *cwd.parents)
     if (p / "dataset").is_dir()
     and (p / "notebook" / "whisper-small").is_dir()),
    None,
)

print(ROOT)

In [ ]:
assert ROOT is not None, f"Start Jupyter from inside myanmar-asr/: {cwd}"

DATASET_DIR = ROOT / "dataset"
MANIFEST_DIR = ROOT / "notebook" / "dataset-mapping"
QUICK_RUN = True
MODEL_ID = "openai/whisper-small"
RUN_DIR = ROOT / "notebook" / "whisper-small" / "run"
CACHE_DIR = ROOT / "notebook" / "whisper-small" / "cache"

print(DATASET_DIR)
print(MANIFEST_DIR)

assert (DATASET_DIR / "NLP Project1" / "audio_files").is_dir()
assert (MANIFEST_DIR / "train_whisper.csv").is_file()

D:\Development\myanmar-asr\dataset
D:\Development\myanmar-asr\notebook\dataset-mapping


In [ ]:
# Check the device cuda

device = "cuda" if torch.cuda.is_available() else "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "cpu"
print({"root": str(ROOT), "device": device, "torch": torch.__version__,
       "transformers": transformers.__version__, "ffmpeg": shutil.which("ffmpeg")})

if device == "cpu":
    print("CPU can check the data pipeline, but full Whisper training needs a GPU in practice.")

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU count:", torch.cuda.device_count())
    print("Current GPU:", torch.cuda.current_device())

elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    print("Apple GPU (MPS) is available")

else:
    print("WARNING: Running on CPU")

{'root': 'D:\\Development\\myanmar-asr', 'device': 'cuda', 'torch': '2.11.0+cu128', 'transformers': '4.57.1', 'ffmpeg': 'C:\\Users\\admin\\AppData\\Local\\Microsoft\\WinGet\\Packages\\Gyan.FFmpeg_Microsoft.Winget.Source_8wekyb3d8bbwe\\ffmpeg-9.0.2-full_build\\bin\\ffmpeg.EXE'}
GPU: NVIDIA GeForce RTX 5060 Ti
GPU count: 1
Current GPU: 0


In [ ]:
import shutil

print("ffmpeg:", shutil.which("ffmpeg"))
print("ffprobe:", shutil.which("ffprobe"))

ffmpeg: C:\Users\admin\AppData\Local\Microsoft\WinGet\Packages\Gyan.FFmpeg_Microsoft.Winget.Source_8wekyb3d8bbwe\ffmpeg-9.0.2-full_build\bin\ffmpeg.EXE
ffprobe: C:\Users\admin\AppData\Local\Microsoft\WinGet\Packages\Gyan.FFmpeg_Microsoft.Winget.Source_8wekyb3d8bbwe\ffmpeg-9.0.2-full_build\bin\ffprobe.EXE


## 3. Load UCSY-only splits and check leakage

The existing manifests already removed missing files, zero-byte files, exact duplicate audio, and paths with conflicting transcripts. They group voice variants and repeated text before splitting. This cell checks those properties again for the UCSY subset. The full test split stays untouched until the final evaluation cell.


In [ ]:
def load_ucsy_split(name):
    with (MANIFEST_DIR / f"{name}_whisper.csv").open(newline="", encoding="utf-8") as handle:
        rows = [row for row in csv.DictReader(handle) if row["source"] == "ucsy"]
    assert rows, f"No UCSY rows in {name}"
    for row in rows:
        path = (DATASET_DIR / row["audio"]).resolve()
        assert path.is_relative_to((DATASET_DIR / "NLP Project1" / "audio_files").resolve()), row["audio"]
        assert path.is_file() and path.stat().st_size > 0, row["audio"]
        assert 0 < float(row["duration_s"]) <= 30, row["" \
        "sample_id"]
    return rows

all_train = load_ucsy_split("train")
all_validation = load_ucsy_split("validation")
all_test = load_ucsy_split("test_ucsy")
for key in ("sha256", "record_group", "text"):
    sets = [set(row[key] for row in rows) for rows in (all_train, all_validation, all_test)]
    assert not (sets[0] & sets[1] or sets[0] & sets[2] or sets[1] & sets[2]), f"{key} crosses splits"

def summary(rows):
    return {"clips": len(rows), "hours": round(sum(float(r["duration_s"]) for r in rows) / 3600, 3),
            "questions": sum(r["side"] == "question" for r in rows),
            "answers": sum(r["side"] == "answer" for r in rows)}

print({"train": summary(all_train), "validation": summary(all_validation), "test": summary(all_test)})
rng = random.Random(20261030)
train_rows, validation_rows = all_train.copy(), all_validation.copy()
if QUICK_RUN:
    rng.shuffle(train_rows)
    rng.shuffle(validation_rows)
    train_rows = train_rows[:256]
    validation_rows = validation_rows[:64]
print("Selected:", summary(train_rows), summary(validation_rows))


{'train': {'clips': 9327, 'hours': 16.697, 'questions': 5231, 'answers': 4096}, 'validation': {'clips': 1177, 'hours': 1.874, 'questions': 830, 'answers': 347}, 'test': {'clips': 1171, 'hours': 1.903, 'questions': 821, 'answers': 350}}
Selected: {'clips': 256, 'hours': 0.456, 'questions': 153, 'answers': 103} {'clips': 64, 'hours': 0.101, 'questions': 48, 'answers': 16}


## 4. Listen to two labels

These examples are a manual spot check, not an automatic proof that all labels are correct. Listen for text/audio mismatch, clipped speech, or a synthetic voice in `audio_files/`. If you find systematic problems, fix the manifest before trusting a model score.


In [ ]:
for side in ("question", "answer"):
    row = next(r for r in train_rows if r["side"] == side)
    print(side, row["sample_id"], row["text"])
    display(Audio(filename=str(DATASET_DIR / row["audio"])))


question ucsy:1950:question:2 အနုပညာနဲ့ ဖျော်ဖြေရေးဘက် ဝါသနာပါရင်ရော ဘယ် Club တွေဝင်လို့ရမလဲ။


answer ucsy:346:answer:1 CST-405 Telecommunication Fundamentals ဖြစ်ပါတယ်။


## 5. Load Whisper and filter labels to its target length

The model downloads on first use. Myanmar transcription is selected explicitly. Rows whose target token sequence exceeds the model's limit are written to a review CSV, with counts printed here.


In [ ]:
processor = WhisperProcessor.from_pretrained(MODEL_ID, language="myanmar", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)
model.generation_config.language = "myanmar"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None
model.config.use_cache = False  # Required when using gradient checkpointing.
model.generation_config.use_cache = True
MAX_LABEL_TOKENS = model.config.max_target_positions

def filter_label_length(rows):
    accepted, rejected = [], []
    for row in rows:
        length = len(processor.tokenizer(row["text"]).input_ids)
        (accepted if length < MAX_LABEL_TOKENS else rejected).append(row)
    return accepted, rejected

train_rows, rejected_train = filter_label_length(train_rows)
validation_rows, rejected_validation = filter_label_length(validation_rows)
assert train_rows and validation_rows
with (RUN_DIR / "long_labels.csv").open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=["sample_id", "audio", "text"])
    writer.writeheader()
    writer.writerows({key: row[key] for key in writer.fieldnames}
                     for row in rejected_train + rejected_validation)
print({"max_label_tokens": MAX_LABEL_TOKENS, "train": summary(train_rows),
       "validation": summary(validation_rows), "long_labels": len(rejected_train) + len(rejected_validation)})


{'max_label_tokens': 448, 'train': {'clips': 245, 'hours': 0.409, 'questions': 153, 'answers': 92}, 'validation': {'clips': 63, 'hours': 0.096, 'questions': 48, 'answers': 15}, 'long_labels': 12}


## 6. Convert selected WAVs to 16 kHz mono

FFmpeg converts each selected input to a deterministic cache path. It runs once per distinct source hash; rerunning this cell reuses completed files. Keep several GB free for a full run. If conversion fails, inspect that file rather than silently dropping it.


In [ ]:
def cache_audio(rows):
    for row in tqdm(rows, desc="Preparing 16 kHz audio"):
        target = CACHE_DIR / f"{row['sha256']}.wav"
        if not target.is_file():
            temporary = CACHE_DIR / f"{row['sha256']}.partial.wav"
            subprocess.run(["ffmpeg", "-nostdin", "-hide_banner", "-loglevel", "error", "-y",
                            "-i", str(DATASET_DIR / row["audio"]), "-ac", "1", "-ar", "16000",
                            "-c:a", "pcm_s16le", str(temporary)], check=True)
            temporary.replace(target)
        info = sf.info(target)
        assert info.samplerate == 16000 and info.channels == 1 and info.frames > 0, target
        row["cache_path"] = str(target)

cache_audio(train_rows + validation_rows)
print("Cache:", CACHE_DIR)


Preparing 16 kHz audio: 100%|██████████| 308/308 [00:00<00:00, 15359.18it/s]

Cache: D:\Development\myanmar-asr\notebook\whisper-small\cache


## 7. Create the training dataset and scoring function

The loader computes Whisper features when a clip is requested. The collator pads text labels and masks padding from the loss. The primary score is CER after removing whitespace, which reduces the effect of inconsistent Myanmar word spacing; raw CER is also reported. Both scores keep punctuation.


In [ ]:
class SpeechDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        audio, rate = sf.read(row["cache_path"], dtype="float32")
        assert rate == 16000 and audio.ndim == 1
        features = processor.feature_extractor(audio, sampling_rate=rate).input_features[0]
        labels = processor.tokenizer(row["text"]).input_ids
        return {"input_features": features, "labels": labels}

class SpeechCollator:
    def __call__(self, features):
        audio_inputs = [{"input_features": item["input_features"]} for item in features]
        batch = processor.feature_extractor.pad(audio_inputs, return_tensors="pt")
        text_inputs = [{"input_ids": item["labels"]} for item in features]
        labels = processor.tokenizer.pad(text_inputs, return_tensors="pt")
        label_ids = labels["input_ids"].masked_fill(labels["attention_mask"].ne(1), -100)
        if (label_ids[:, 0] == processor.tokenizer.bos_token_id).all().item():
            label_ids = label_ids[:, 1:]
        batch["labels"] = label_ids
        return batch

def no_spaces(text):
    return "".join(unicodedata.normalize("NFC", text).split())

def compute_metrics(result):
    predictions = result.predictions[0] if isinstance(result.predictions, tuple) else result.predictions
    labels = np.where(result.label_ids == -100, processor.tokenizer.pad_token_id, result.label_ids)
    hypotheses = processor.tokenizer.batch_decode(predictions, skip_special_tokens=True)
    references = processor.tokenizer.batch_decode(labels, skip_special_tokens=True)
    return {"cer_no_space": 100 * jiwer.cer([no_spaces(x) for x in references],
                                              [no_spaces(x) for x in hypotheses]),
            "cer_raw": 100 * jiwer.cer(references, hypotheses)}

train_dataset = SpeechDataset(train_rows)
validation_dataset = SpeechDataset(validation_rows)
sample = train_dataset[0]
assert sample["input_features"].ndim == 2 and sample["labels"]
assert SpeechCollator()([sample])["labels"].shape[0] == 1
print("One sample loaded and collated successfully.")


You're using a WhisperTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


One sample loaded and collated successfully.


## 8. Configure a short check or a full training run

The quick run uses 20 optimizer steps. Full mode trains for three epochs and chooses the checkpoint with the lowest validation CER. CUDA uses mixed precision; other devices use full precision. Training uses one clip per GPU batch with 16 accumulation steps (effective batch size 16), and validation uses one clip per GPU batch to reduce peak memory. If the display goes black during training, check the Windows System log for a GPU driver reset before retrying.


In [ ]:
interval = 20 if QUICK_RUN else 500
training_args = Seq2SeqTrainingArguments(
    output_dir=str(RUN_DIR),
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=1e-5,
    warmup_steps=5 if QUICK_RUN else 100,
    num_train_epochs=3,
    max_steps=20 if QUICK_RUN else -1,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    fp16=(device == "cuda"),
    optim="adamw_torch",
    eval_strategy="steps",
    eval_steps=interval,
    save_strategy="steps",
    save_steps=interval,
    save_total_limit=2,
    logging_steps=5 if QUICK_RUN else 50,
    predict_with_generate=True,
    generation_max_length=MAX_LABEL_TOKENS,
    eval_accumulation_steps=4,
    dataloader_num_workers=0,  # Keeps notebook data loading reliable on Windows.
    load_best_model_at_end=True,
    metric_for_best_model="cer_no_space",
    greater_is_better=False,
    report_to=[],
    push_to_hub=False,
    seed=20261030,
)
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    data_collator=SpeechCollator(),
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
)
print("Output:", RUN_DIR)


Output: D:\Development\myanmar-asr\notebook\whisper-small\run


## 9. Measure the pretrained baseline, then fine-tune

The baseline uses the exact same validation clips and scoring function as training. After fine-tuning, the best checkpoint is saved with the processor. Compare the printed metrics before deciding whether the model improved.


In [13]:
baseline = trainer.evaluate(metric_key_prefix="baseline")
print("Pretrained baseline:", baseline)
(RUN_DIR / "baseline_metrics.json").write_text(json.dumps(baseline, indent=2), encoding="utf-8")
train_result = trainer.train()
print("Training:", train_result.metrics)
best_dir = RUN_DIR / "best_model"
trainer.save_model(str(best_dir))
processor.save_pretrained(str(best_dir))
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Validation after training:", trainer.evaluate(metric_key_prefix="validation"))


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Pretrained baseline: {'baseline_loss': 2.3270010948181152, 'baseline_model_preparation_time': 0.0031, 'baseline_cer_no_space': 264.5273899033298, 'baseline_cer_raw': 299.14257716805486, 'baseline_runtime': 148.0336, 'baseline_samples_per_second': 0.426, 'baseline_steps_per_second': 0.216}


Step,Training Loss,Validation Loss


AcceleratorError: CUDA error: unknown error
Search for `cudaErrorUnknown' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


## 10. Evaluate the held-out UCSY test set once

This cell is skipped in quick mode. In full mode, run it after training and checkpoint selection. It stores every reference and prediction for error review. The test clips are human-speech candidates from `audio_files/`; synthetic clips are absent.


In [ ]:
if QUICK_RUN:
    print("Test skipped in QUICK_RUN. Switch to full mode and rerun from the top.")
else:
    test_rows, rejected_test = filter_label_length(all_test)
    cache_audio(test_rows)
    result = trainer.predict(SpeechDataset(test_rows), metric_key_prefix="test_ucsy")
    print("Held-out test:", result.metrics, "long labels excluded:", len(rejected_test))
    predictions = result.predictions[0] if isinstance(result.predictions, tuple) else result.predictions
    labels = np.where(result.label_ids == -100, processor.tokenizer.pad_token_id, result.label_ids)
    hypotheses = processor.tokenizer.batch_decode(predictions, skip_special_tokens=True)
    references = processor.tokenizer.batch_decode(labels, skip_special_tokens=True)
    with (RUN_DIR / "test_predictions.csv").open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=["sample_id", "audio", "side", "reference", "prediction"])
        writer.writeheader()
        writer.writerows({"sample_id": row["sample_id"], "audio": row["audio"], "side": row["side"],
                          "reference": ref, "prediction": pred}
                         for row, ref, pred in zip(test_rows, references, hypotheses))
    (RUN_DIR / "test_metrics.json").write_text(json.dumps(result.metrics, indent=2), encoding="utf-8")
    print("Saved:", RUN_DIR / "test_predictions.csv")


## 11. Transcribe one clip with the trained model

This last cell demonstrates inference using the best loaded checkpoint. It uses one validation clip, so it does not consult the held-out test set. The saved model is in `asr_work/ucsy_whisper_full/best_model/` after a full run.


In [ ]:
row = validation_rows[0]
audio, rate = sf.read(row["cache_path"], dtype="float32")
inputs = processor(audio, sampling_rate=rate, return_tensors="pt")
inputs = {key: value.to(trainer.model.device) for key, value in inputs.items()}
with torch.no_grad():
    output_ids = trainer.model.generate(**inputs, max_new_tokens=256)
print("Reference:", row["text"])
print("Prediction:", processor.batch_decode(output_ids, skip_special_tokens=True)[0])
